# 01 - Data Exploration

Explore the BDD100K dataset and auto-labeling pipeline for Constellation.

**Goals:**
- Load and visualize sample driving images
- Test the auto-labeling pipeline (YOLOv8 + MobileSAM)
- Plot class distributions from auto-generated labels
- Compute dataset statistics

In [ ]:
import os
import sys
import json
import requests
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# API base URL
API_BASE = "http://localhost:8000"

# Data directories
DATA_DIR = PROJECT_ROOT / "data" / "bdd100k"
IMAGES_DIR = DATA_DIR / "images" / "100k" / "train"

print(f"Project root: {PROJECT_ROOT}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Images exist: {IMAGES_DIR.exists()}")
print(f"Image count: {len(list(IMAGES_DIR.glob('*.jpg'))) if IMAGES_DIR.exists() else 0}")

## Fetch Data from API

Get images and their auto-generated labels from the Constellation backend.

In [ ]:
# Fetch images from API
response = requests.get(f"{API_BASE}/api/images?limit=100")
data = response.json()

print(f"Total images in database: {data['total']}")
print(f"\nSample images:")
for img in data['images'][:5]:
    print(f"  - {img['filename']} ({img['width']}x{img['height']}) - {img['status']}")

In [ ]:
# Fetch labels for each image
images_with_labels = []

for img in data['images'][:20]:  # First 20 images
    detail_resp = requests.get(f"{API_BASE}/api/images/{img['id']}")
    detail = detail_resp.json()
    if detail['labels']:
        images_with_labels.append(detail)

print(f"Images with auto-labels: {len(images_with_labels)}")

## Class Distribution

Analyze the distribution of detected object categories from auto-labeling.

In [ ]:
# Count object categories from auto-labels
category_counts = Counter()
confidences_by_class = {}
boxes_per_image = []

for item in images_with_labels:
    for label in item['labels']:
        class_names = label['data'].get('class_names', [])
        confs = label['data'].get('confidences', [])
        boxes_per_image.append(len(class_names))
        
        for cls, conf in zip(class_names, confs):
            category_counts[cls] += 1
            if cls not in confidences_by_class:
                confidences_by_class[cls] = []
            confidences_by_class[cls].append(conf)

print("Detected object categories:")
for cat, count in category_counts.most_common():
    avg_conf = np.mean(confidences_by_class[cat])
    print(f"  {cat}: {count} detections (avg conf: {avg_conf:.1%})")

print(f"\nTotal detections: {sum(category_counts.values())}")
if boxes_per_image:
    print(f"Average detections per image: {np.mean(boxes_per_image):.1f}")

In [ ]:
# Plot class distribution
if category_counts:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Category distribution bar chart
    categories = list(category_counts.keys())
    counts = list(category_counts.values())
    
    ax1 = axes[0]
    colors = plt.cm.Set3(np.linspace(0, 1, len(categories)))
    bars = ax1.barh(categories, counts, color=colors)
    ax1.set_xlabel('Count')
    ax1.set_title('Object Category Distribution (Auto-labeled)')
    ax1.invert_yaxis()
    
    for bar, count in zip(bars, counts):
        ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                 f'{count}', va='center', fontsize=9)
    
    # Confidence distribution
    ax2 = axes[1]
    all_confs = [c for confs in confidences_by_class.values() for c in confs]
    ax2.hist(all_confs, bins=20, color='coral', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Confidence Score')
    ax2.set_ylabel('Count')
    ax2.set_title('Detection Confidence Distribution')
    ax2.axvline(np.mean(all_confs), color='red', linestyle='--', 
                label=f'Mean: {np.mean(all_confs):.2f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No label data available. Run auto-labeling first.")

## Sample Images with Auto-Labels

Visualize sample images with their YOLOv8 detections.

In [ ]:
# Color palette for categories
CATEGORY_COLORS = {
    'car': '#FF6B6B',
    'truck': '#4ECDC4',
    'bus': '#45B7D1',
    'person': '#96CEB4',
    'bicycle': '#DDA0DD',
    'motorcycle': '#98D8C8',
    'traffic light': '#F7DC6F',
    'stop sign': '#BB8FCE',
}

def visualize_auto_labels(image_data, ax=None):
    """Display an image with its auto-generated labels."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Load image
    img_path = Path(image_data['image']['file_path'])
    if not img_path.exists():
        # Try relative path
        img_path = PROJECT_ROOT / 'data' / 'bdd100k' / 'images' / '100k' / 'train' / image_data['image']['filename']
    
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(image_data['image']['filename'], fontsize=10)
    ax.axis('off')
    
    # Draw bounding boxes from labels
    for label in image_data['labels']:
        boxes = label['data'].get('boxes', [])
        class_names = label['data'].get('class_names', [])
        confidences = label['data'].get('confidences', [])
        
        for box, cls, conf in zip(boxes, class_names, confidences):
            x1, y1, x2, y2 = box
            width, height = x2 - x1, y2 - y1
            
            color = CATEGORY_COLORS.get(cls, '#FFFFFF')
            
            rect = patches.Rectangle(
                (x1, y1), width, height,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            
            ax.text(x1, y1 - 5, f"{cls} {conf:.0%}", fontsize=8, color=color,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    
    return ax

In [ ]:
# Display sample images with auto-labels
if images_with_labels:
    n_samples = min(6, len(images_with_labels))
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for ax, img_data in zip(axes, images_with_labels[:n_samples]):
        try:
            visualize_auto_labels(img_data, ax)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {e}", ha='center', va='center')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No labeled images found. Run auto-labeling via the frontend or API first.")

## Test Auto-Labeler Directly

Run the auto-labeling pipeline on a sample image.

In [ ]:
# Test auto-labeler directly
try:
    from data_engine.auto_labeler import AutoLabeler
    
    # Initialize auto-labeler (without SAM for speed)
    labeler = AutoLabeler(use_sam=False, conf_threshold=0.3)
    
    # Get sample image
    image_files = list(IMAGES_DIR.glob("*.jpg"))
    if image_files:
        test_image = image_files[0]
        print(f"Testing auto-labeler on: {test_image.name}")
        
        # Run detection
        results = labeler.label_image(test_image)
        
        print(f"\nDetected {len(results['boxes'])} objects:")
        for cls_name, conf, box in zip(results['class_names'], 
                                        results['confidences'], 
                                        results['boxes']):
            print(f"  - {cls_name}: {conf:.1%} at [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")
        
        # Visualize
        fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        img = Image.open(test_image)
        ax.imshow(img)
        ax.set_title(f"Auto-Labeled: {test_image.name}")
        ax.axis('off')
        
        for box, cls_name, conf in zip(results['boxes'], results['class_names'], results['confidences']):
            x1, y1, x2, y2 = box
            color = CATEGORY_COLORS.get(cls_name, 'lime')
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{cls_name} {conf:.0%}", fontsize=10, color=color,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        
        plt.tight_layout()
        plt.show()
    else:
        print("No images found.")

except ImportError as e:
    print(f"Auto-labeler dependencies not installed: {e}")
    print("Install with: pip install ultralytics torch torchvision")

## Summary

### Phase 1 Complete: Data Engine

**What we built:**
- Auto-labeling pipeline with YOLOv8 detection + MobileSAM segmentation
- FastAPI backend with PostgreSQL storage
- React frontend for viewing and labeling images
- Data ingestion from BDD100K dataset

**Detection classes:**
- car, truck, bus, person, bicycle, motorcycle, traffic light, stop sign

### Next Steps
1. Proceed to model training: `02_model_training.ipynb`
2. Build HydraNet multi-task model
3. Train on BDD100K subset